# 📥 Chapter 02: Spatial ETL, Ingestion & Automated Topology Cleaning
## Automated Detection and Repair of Corrupted Building Geometries

### 🎯 The Mission: Why We Are Doing This
Spatial datasets from digitizing surveys, OpenStreetMap, or CAD conversions frequently contain topological errors:
1. **Self-intersecting "figure-8" polygons** (borders crossing themselves).
2. **Unclosed boundary rings**.
3. **Micro-sliver coordinate jitter**.

If you insert invalid geometries into PostGIS, subsequent spatial operations (`ST_Intersection`, `ST_Buffer`) will crash with `GEOSTopologyException`. A Spatial DBA must build automated cleaning and quarantine pipelines.

---

### 🔑 Spatial DBA Keywords Decoded

| Keyword / Function | Plain-English Explanation | Production Note |
| :--- | :--- | :--- |
| **`ST_IsValid(geom)`** | Returns `TRUE` if a polygon obeys OGC topological rules; `FALSE` if broken. | Used as a filter and table check constraint. |
| **`ST_IsValidReason(geom)`** | Returns a diagnostic string explaining *why* geometry failed (e.g. `Self-intersection [809125 614125]`). | Logs exact error locations for survey audits. |
| **`ST_MakeValid(geom)`** | Automatically repairs self-intersections and invalid rings using GEOS algorithms. | Fixes figure-8 polygons into valid MultiPolygons. |
| **`ST_SnapToGrid(geom, tol)`** | Snaps coordinates to a grid (e.g. $1	ext{mm} = 0.001	ext{m}$) to remove micro-jitter. | Prevents precision floating-point errors. |
| **`ST_CollectionExtract(geom, type)`** | Extracts only Polygons (3), Lines (2), or Points (1) from messy `GeometryCollections`. | Enforces strict geometry types. |

In [ ]:
from config_db import execute_sql, query_df, query_gdf, preview_map

sql_etl = """
DROP TABLE IF EXISTS staging_buildings CASCADE;
CREATE TABLE staging_buildings (
    building_id SERIAL PRIMARY KEY,
    building_tag VARCHAR(50),
    raw_geom GEOMETRY(Geometry, 32630)
);

INSERT INTO staging_buildings (building_tag, raw_geom)
VALUES 
    ('residential', ST_GeomFromText('POLYGON((809000 614000, 809020 614000, 809020 614030, 809000 614030, 809000 614000))', 32630)),
    ('commercial', ST_GeomFromText('POLYGON((809100 614100, 809150 614150, 809150 614100, 809100 614150, 809100 614100))', 32630)),
    ('residential', ST_GeomFromText('POLYGON((809200 614200, 809200.0001 614200, 809210 614220, 809200 614220, 809200 614200))', 32630));

DROP TABLE IF EXISTS clean_buildings CASCADE;
CREATE TABLE clean_buildings AS
SELECT 
    building_id,
    building_tag,
    ST_Multi(ST_CollectionExtract(ST_MakeValid(ST_SnapToGrid(raw_geom, 0.001)), 3))::GEOMETRY(MultiPolygon, 32630) AS geom,
    NOT ST_IsValid(raw_geom) AS was_repaired
FROM staging_buildings;

CREATE INDEX idx_clean_bld_geom ON clean_buildings USING GIST(geom);
"""

execute_sql(sql_etl)
print("✅ Staging data processed, repaired, and validated.")

In [ ]:
# Inspect Cleaning Results
df_clean = query_df("""
SELECT building_id, building_tag, was_repaired, ST_IsValid(geom) AS is_now_valid, ST_Area(geom) AS area_m2
FROM clean_buildings;
""")

df_clean

In [ ]:
# Interactive Map Preview of Repaired Buildings
gdf_clean = query_gdf("""
SELECT building_id, building_tag, was_repaired, geom FROM clean_buildings;
""")

preview_map(gdf_clean, tooltip_cols=["building_id", "building_tag", "was_repaired"])